# Proyecto Integrador: Trading Bot Cuantitativo & Machine Learning
## Simulación de Hiperparámetros, Métricas Cuantitativas y Test de Diebold-Mariano (1995)

Este notebook documenta la fundamentación matemática, entrenamiento del clasificador **Random Forest**, comparación de hiperparámetros y validación estadística rigurosa mediante el **Test de Diebold-Mariano con corrección de Harvey-Leybourne-Newbold (HLN, 1997)**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from core.data_feed import fetch_historical_data
from core.indicators import compute_all_indicators
from core.ml_engine import prepare_features_and_target, simulate_hyperparameters_grid, diebold_mariano_test
from core.monte_carlo import run_monte_carlo_simulation

### 1. Ingesta de Datos y Cálculo de Indicadores Técnicos
Se calculan: MACD, RSI, Medias Móviles de Regresión, Oscilador Estocástico, SAR Parabólico y ATR.

In [ ]:
df_raw = fetch_historical_data('BTCUSDT', period='60d', interval='1h')
clean_df, X, y_class, y_cont = prepare_features_and_target(df_raw)
print(f'Muestras limpias preparadas: {len(clean_df)}')
clean_df[['close', 'rsi_14', 'macd', 'lr_slope', 'parabolic_sar', 'atr_14']].tail()

### 2. Simulación de Hiperparámetros (Grid Search con TimeSeriesSplit)
Evaluación de RMSE, MAE, SMAPE, MAPE, Variance Score ($R^2$), Accuracy y F1-Score.

In [ ]:
results = simulate_hyperparameters_grid(X, y_class, y_cont)
sim_df = pd.DataFrame(results['simulation_table'])
sim_df

### 3. Test Estadístico de Diebold-Mariano (Random Forest vs KNN Benchmark)
$$H_0: E[d_t] = 0 \quad \text{vs} \quad H_1: E[d_t] \neq 0$$
donde $d_t = L(e_{1,t}) - L(e_{2,t})$ y $DM^* = DM \times \sqrt{\frac{T + 1 - 2h + h(h-1)/T}{T}}$.

In [ ]:
from core.ml_engine import QuantitativeMLEngine
engine = QuantitativeMLEngine()
metrics = engine.train_and_evaluate(df_raw)
dm = metrics['diebold_mariano']
print(f'Estadístico Diebold-Mariano (HLN corregido): {dm["dm_stat"]:.4f}')
print(f'p-valor: {dm["p_value"]:.4f}')
print(f'Veredicto: {dm["conclusion"]}')

### 4. Simulación Monte Carlo (1,000 Caminos por Movimiento Browniano)
Evaluación de la distribución estocástica y probabilidad de alcanzar Take Profit antes de Stop Loss.

In [ ]:
current_price = float(df_raw['close'].iloc[-1])
sl = current_price * 0.97
tp = current_price * 1.05
mc = run_monte_carlo_simulation(df_raw, current_price, sl, tp, num_simulations=1000)
print(mc['summary'])